In [21]:
import os
import json
import pandas as pd

# Carpeta con los archivos JSON
ruta_carpeta = "./open-data/data/events"  # Ajuste de ruta para evitar errores en diferentes sistemas operativos

# Lista para almacenar los datos
datos_disparos = []


# Recorrer los archivos JSON en la carpeta
for archivo in os.listdir(ruta_carpeta):
    if archivo.endswith(".json"):
        with open(os.path.join(ruta_carpeta, archivo), encoding="utf-8") as f:
            eventos = json.load(f)
            
            # Filtrar solo eventos de tipo "Shot"
            
            for evento in eventos:  # eventos = lista cargada desde el JSON de StatsBomb
                if evento["type"]["name"] == "Shot":  # Solo procesar eventos de tipo 'Shot'
                    datos_disparos.append({
                        "event_id": evento["id"],
                        "match_id": archivo.replace(".json", ""),
                        "player": evento["player"]["name"] if "player" in evento else None,
                        "team": evento["team"]["name"] if "team" in evento else None,
                        "minute": evento.get("minute"),
                        "play_pattern": evento["play_pattern"]['name'] if "play_pattern" in evento else None,
                        
                        # Ubicación inicial del disparo
                        "location_x": evento["location"][0] if "location" in evento else None,
                        "location_y": evento["location"][1] if "location" in evento else None,
                        
                
                        
                        # Variables booleanas
                        "under_pressure": evento.get("under_pressure", False),
                        "open_goal": evento["shot"].get("open_goal", False) if "shot" in evento else False,
                        "aerial_won": evento["shot"].get("aerial_won", False) if "shot" in evento else False,
                        "follows_dribble": evento["shot"].get("follows_dribble", False) if "shot" in evento else False,
                        "first_time": evento["shot"].get("first_time", False) if "shot" in evento else False,
                        
                        
                        # Técnica, parte del cuerpo, tipo de tiro, resultado
                        "shot_technique": evento["shot"]["technique"]["name"] if "shot" in evento and "technique" in evento["shot"] else None,
                        "shot_body_part": evento["shot"]["body_part"]["name"] if "shot" in evento and "body_part" in evento["shot"] else None,
                        "shot_type": evento["shot"]["type"]["name"] if "shot" in evento and "type" in evento["shot"] else None,
                        "is_goal": True if ("shot" in evento and "outcome" in evento["shot"] and evento["shot"]["outcome"]["name"] == "Goal") else False,
                        
                        # Expected Goals (xG)
                        "statsbomb_xg": evento["shot"].get("statsbomb_xg") if "shot" in evento else None,
                        
                        # Freeze frame (lista de jugadores en el momento del disparo)
                        "freeze_frame": evento["shot"].get("freeze_frame") if "shot" in evento else None,
                        "key_pass_id": evento["shot"].get("key_pass_id") if "shot" in evento else None
                    })

                  
# Convertir a DataFrames
df_disparos = pd.DataFrame(datos_disparos)


# Guardar como CSV
df_disparos.to_csv("shots_data.csv", index=False, encoding="utf-8")




In [14]:
import pandas as pd

# Crear lista para los freeze frames expandidos
datos_freeze = []

for _, row in df_disparos.iterrows():
    event_id = row["event_id"]
    match_id = row["match_id"]
    
    freeze_frame = row["freeze_frame"]
    if freeze_frame and isinstance(freeze_frame, list):
        for jugador in freeze_frame:
            datos_freeze.append({
                "event_id": event_id,
                "match_id": match_id,
                "location_x": jugador["location"][0] if "location" in jugador else None,
                "location_y": jugador["location"][1] if "location" in jugador else None,
                "teammate": jugador.get("teammate", None),
                "position": jugador["position"]['id'] if "position" in jugador else None,
                "player": jugador["player"]['name'] if "player" in jugador else None,
                
                
            })

# Crear dataframe final
df_freeze = pd.DataFrame(datos_freeze)

# Convertir a DataFrame final
df_freeze.to_csv("freeze_frames.csv", index=False, encoding="utf-8")

In [15]:
df_freeze.head()

,event_id,match_id,location_x,location_y,teammate,position,player
0,becd7956-ce44-479e-8fc9-16a2d1f1f349,15946,91.1,50.2,False,15,Manuel Alejandro García Sánchez
1,becd7956-ce44-479e-8fc9-16a2d1f1f349,15946,93.2,43.7,False,10,Daniel Alejandro Torres Rojas
2,becd7956-ce44-479e-8fc9-16a2d1f1f349,15946,82.9,59.5,False,16,Jonathan Rodríguez Menéndez
3,becd7956-ce44-479e-8fc9-16a2d1f1f349,15946,103.0,44.8,True,22,Luis Alberto Suárez Díaz
4,becd7956-ce44-479e-8fc9-16a2d1f1f349,15946,95.4,53.4,True,12,Sergi Roberto Carnicer


In [22]:
import pandas as pd

# Variables categóricas que quieres codificar
categoricas = ["shot_technique", "shot_body_part", "shot_type","play_pattern"]

# One-Hot Encoding integrado en el dataset
df_disparos_onehot = pd.get_dummies(df_disparos, columns=categoricas, prefix=categoricas)

# Guardar dataset final
df_disparos_onehot.to_csv("disparos_final.csv", index=False, encoding="utf-8")

In [23]:
df_disparos_onehot.head()

,event_id,match_id,player,team,minute,location_x,location_y,under_pressure,open_goal,aerial_won,...,shot_type_Penalty,play_pattern_From Corner,play_pattern_From Counter,play_pattern_From Free Kick,play_pattern_From Goal Kick,play_pattern_From Keeper,play_pattern_From Kick Off,play_pattern_From Throw In,play_pattern_Other,play_pattern_Regular Play
0,becd7956-ce44-479e-8fc9-16a2d1f1f349,15946,Lionel Andrés Messi Cuccittini,Barcelona,2,111.5,52.9,False,False,False,...,False,False,False,False,False,False,False,True,False,False
1,9107d374-2942-4876-a14f-1b9f86901c15,15946,Jordi Alba Ramos,Barcelona,5,113.9,26.4,False,False,False,...,False,False,False,False,False,False,False,False,False,True
2,ddd194ca-08fb-43d0-87c2-33647f975f9f,15946,Lionel Andrés Messi Cuccittini,Barcelona,15,93.7,34.7,False,False,False,...,False,False,False,False,False,True,False,False,False,False
3,86596ddb-d824-4e5e-b18c-b4442e9ce7cf,15946,Rubén Sobrino Pozuelo,Deportivo Alavés,16,109.2,39.1,True,False,True,...,False,False,False,False,False,False,False,False,False,True
4,3ed2b107-be17-42d5-9d1b-25006a0e55cb,15946,Luis Alberto Suárez Díaz,Barcelona,18,107.8,24.7,False,False,False,...,False,True,False,False,False,False,False,False,False,False
